# Animal Movement Demo

Demonstrates the acoustic telemetry state-space model from Lavender et al. on synthetic data.
The state is $(x, y, \phi)$ — 2D position in meters plus heading — and evolves as a correlated
random walk.  Observations are binary detections at acoustic receivers.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpyro
import numpyro.distributions as dist

import dynestyx as dsx
from dynestyx.inference.filter_configs import PFConfig
from dynestyx.inference.filters import Filter
from dynestyx.inference.smoother_configs import PFSmootherConfig
from dynestyx.inference.smoothers import Smoother

from animal_movement_paper.models import (
    TelemetryObservationModel,
    animal_movement_model,
)
from animal_movement_paper.distributions import (
    step_length_sample,
    turning_angle_sample,
    wrap_angle,
)

key = jax.random.PRNGKey(42)

## 1. Study area and receivers

A synthetic 10 km × 10 km lake with 8 acoustic receivers on a grid.

In [ ]:
LAKE_SIZE = 10_000.0  # metres

# 8 receivers on a 2x4 interior grid
rx = jnp.array([2500.0, 5000.0, 7500.0, 2500.0, 5000.0, 7500.0, 3500.0, 6500.0])
ry = jnp.array([2500.0, 2500.0, 2500.0, 7500.0, 7500.0, 7500.0, 5000.0, 5000.0])
receiver_locations = jnp.stack([rx, ry], axis=-1)
n_receivers = receiver_locations.shape[0]
print(f"Receivers: {n_receivers}")

## 2. Simulate ground-truth trajectory

Forward-simulate the correlated random walk for 200 steps, then generate binary detections
from the logistic detection probability model.

In [ ]:
N_STEPS = 200

def simulate_trajectory(key, n_steps=N_STEPS):
    k0, k_traj = jax.random.split(key)
    # Start near the centre of the lake heading east
    init_state = jnp.array([5000.0, 5000.0, 0.0])

    def step(state, key):
        k1, k2 = jax.random.split(key)
        d = step_length_sample(k1)
        dphi = turning_angle_sample(k2)
        phi_new = state[2] + dphi
        x_new = state[0] + d * jnp.cos(phi_new)
        y_new = state[1] + d * jnp.sin(phi_new)
        # Reflect off lake boundaries (soft bounce)
        x_new = jnp.clip(x_new, 0.0, LAKE_SIZE)
        y_new = jnp.clip(y_new, 0.0, LAKE_SIZE)
        new_state = jnp.array([x_new, y_new, phi_new])
        return new_state, new_state

    keys = jax.random.split(k_traj, n_steps)
    _, states = jax.lax.scan(step, init_state, keys)
    return jnp.concatenate([init_state[None], states], axis=0)  # (n_steps+1, 3)


def simulate_observations(key, states):
    obs_model = TelemetryObservationModel(receiver_locations=receiver_locations)

    def obs_at_t(key, state):
        return obs_model(state, None, 0.0).sample(key)

    keys = jax.random.split(key, states.shape[0])
    return jax.vmap(obs_at_t)(keys, states)


k1, k2 = jax.random.split(key)
true_states = simulate_trajectory(k1)
obs_values = simulate_observations(k2, true_states)

print(f"Trajectory shape: {true_states.shape}")
print(f"Observations shape: {obs_values.shape}")
print(f"Detection rate: {obs_values.mean():.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(true_states[:, 0], true_states[:, 1], 'k-', alpha=0.5, linewidth=0.8, label='True trajectory')
ax.scatter(true_states[0, 0], true_states[0, 1], color='green', s=80, zorder=5, label='Start')
ax.scatter(true_states[-1, 0], true_states[-1, 1], color='red', s=80, zorder=5, label='End')
ax.scatter(receiver_locations[:, 0], receiver_locations[:, 1],
           marker='^', color='blue', s=100, zorder=5, label='Receivers')
ax.set_xlim(0, LAKE_SIZE)
ax.set_ylim(0, LAKE_SIZE)
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Simulated lake trout trajectory')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

## 3. Particle filter

Build the dynestyx model and run the bootstrap particle filter with 1,000 particles.

In [ ]:
obs_times = jnp.arange(N_STEPS + 1, dtype=float)

pf_config = PFConfig(
    n_particles=1_000,
    record_filtered_particles=True,
    record_filtered_log_weights=True,
    record_filtered_states_mean=True,
)

def model(obs_times=None, obs_values=None):
    dynamics = animal_movement_model(
        receiver_locations=receiver_locations,
        study_area_bounds=jnp.array([0.0, 0.0, LAKE_SIZE, LAKE_SIZE]),
    )
    return dsx.sample("f", dynamics, obs_times=obs_times, obs_values=obs_values)

pf_key = jax.random.PRNGKey(7)
with Filter(filter_config=pf_config):
    pf_trace = numpyro.infer.Predictive(model, num_samples=1)(
        pf_key, obs_times=obs_times, obs_values=obs_values
    )

filtered_particles = pf_trace["f_filtered_particles"][0]  # (T, N, 3)
filtered_log_weights = pf_trace["f_filtered_log_weights"][0]  # (T, N)
filtered_means = pf_trace["f_filtered_states_mean"][0]  # (T, 3)
print(f"Filtered particles shape: {filtered_particles.shape}")

In [ ]:
def weighted_mean(particles, log_weights):
    weights = jax.nn.softmax(log_weights, axis=-1)  # (T, N)
    return jnp.einsum('tn,tnd->td', weights, particles)  # (T, 3)

pf_mean = weighted_mean(filtered_particles, filtered_log_weights)

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(true_states[:, 0], true_states[:, 1], 'k-', alpha=0.4, linewidth=1.0, label='True')
ax.plot(pf_mean[:, 0], pf_mean[:, 1], 'b-', alpha=0.8, linewidth=1.5, label='PF mean')
# Particle cloud at step 100
t_show = 100
w = jax.nn.softmax(filtered_log_weights[t_show])
ax.scatter(filtered_particles[t_show, :, 0], filtered_particles[t_show, :, 1],
           s=1.0, alpha=0.3, color='cornflowerblue', label=f'Particles at t={t_show}')
ax.scatter(receiver_locations[:, 0], receiver_locations[:, 1],
           marker='^', color='red', s=80, zorder=5, label='Receivers')
ax.set_xlim(0, LAKE_SIZE)
ax.set_ylim(0, LAKE_SIZE)
ax.set_xlabel('x (m)')
ax.set_ylabel('y (m)')
ax.set_title('Particle filter')
ax.legend(loc='upper right', markerscale=3)
plt.tight_layout()
plt.show()

## 4. Particle smoother

The two-filter smoother improves estimates by incorporating future observations.

In [ ]:
smoother_config = PFSmootherConfig(
    n_particles=1_000,
    record_smoothed_particles=True,
    record_smoothed_states_mean=True,
)

smoother_key = jax.random.PRNGKey(13)
with Smoother(smoother_config=smoother_config):
    smoother_trace = numpyro.infer.Predictive(model, num_samples=1)(
        smoother_key, obs_times=obs_times, obs_values=obs_values
    )

smoothed_particles = smoother_trace["f_smoothed_particles"][0]  # (T, N, 3)
smoothed_means = smoother_trace["f_smoothed_states_mean"][0]  # (T, 3)
print(f"Smoothed particles shape: {smoothed_particles.shape}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, mean, title in zip(axes, [pf_mean, smoothed_means], ['Particle filter', 'Particle smoother']):
    ax.plot(true_states[:, 0], true_states[:, 1], 'k-', alpha=0.4, linewidth=1.0, label='True')
    ax.plot(mean[:, 0], mean[:, 1], 'b-', alpha=0.9, linewidth=1.5, label='Posterior mean')
    ax.scatter(receiver_locations[:, 0], receiver_locations[:, 1],
               marker='^', color='red', s=80, zorder=5, label='Receivers')
    ax.set_xlim(0, LAKE_SIZE)
    ax.set_ylim(0, LAKE_SIZE)
    ax.set_xlabel('x (m)')
    ax.set_ylabel('y (m)')
    ax.set_title(title)
    ax.legend(loc='upper right')

plt.tight_layout()
plt.show()

## 5. Position error over time

In [ ]:
pf_error = jnp.sqrt(jnp.sum((pf_mean[:, :2] - true_states[:, :2])**2, axis=-1))
sm_error = jnp.sqrt(jnp.sum((smoothed_means[:, :2] - true_states[:, :2])**2, axis=-1))

fig, ax = plt.subplots(figsize=(9, 3))
ax.plot(pf_error, label=f'Filter  (mean {pf_error.mean():.0f} m)', alpha=0.8)
ax.plot(sm_error, label=f'Smoother (mean {sm_error.mean():.0f} m)', alpha=0.8)
ax.set_xlabel('Time step')
ax.set_ylabel('Position error (m)')
ax.set_title('Positional error: filter vs smoother')
ax.legend()
plt.tight_layout()
plt.show()

print(f"Mean filter error:   {float(pf_error.mean()):.1f} m")
print(f"Mean smoother error: {float(sm_error.mean()):.1f} m")